# Notebook 13: Multi-Agent Collaboration with LangGraph

**🎯 Goal:** Learn how to build sophisticated AI systems where multiple, specialized agents collaborate to solve a complex problem, orchestrated by LangGraph.

## 🧩 Concept Overview

Many complex tasks are too difficult for a single agent. Just like in a human team, you can achieve better results by having multiple specialized agents work together. For example, to write a report, you might have:

- A **Researcher Agent** to gather information.
- A **Writer Agent** to draft the report based on the research.
- A **Reviewer Agent** to critique the draft and suggest improvements.

**LangGraph** is the perfect tool for orchestrating these multi-agent workflows. You can define a 'supervisor' or 'router' that directs the flow of the task between the different agents, creating a collaborative AI team.

## 🖼️ Visual Diagram: A Multi-Agent Team

```
                 +--------------+
User Task ---->  |  Supervisor  | (Routes task to the right agent)
                 +------+-------+
                        |         
          /-------------+------------\
          |             |            |
          v             v            v
  +--------------+ +--------------+ +--------------+
  | Researcher   | |   Writer     | |   Reviewer   |
  +--------------+ +--------------+ +--------------+
          ^             ^            ^
          |             |            |
          \-------------+------------/
                        |
                 (Returns to Supervisor to decide next step)

```

## ⚙️ Setup

We'll need the standard LangGraph and OpenAI libraries.

In [ ]:
import os
from dotenv import load_dotenv
from typing import TypedDict, Annotated, List, Union
import operator
from langchain_core.agents import AgentAction, AgentFinish
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langgraph.graph import StateGraph, END

load_dotenv()
llm = ChatOpenAI(model="gpt-4o")

## 1. Defining the State and Agents

First, we define our shared `GraphState`. This is like the 'whiteboard' that all agents will read from and write to. Then, we'll create simple functions that represent our specialized agents.

In [ ]:
# The shared state for our graph
class GraphState(TypedDict):
    task: str
    research_notes: str
    draft: str
    review: str
    # The 'next' field will be used by the supervisor to decide which agent to call next
    next: str

# Agent 1: Researcher
def researcher_agent(state: GraphState) -> GraphState:
    print("---> In Researcher Agent")
    task = state.get("task")
    notes = f"Research notes on '{task}': AI is growing fast. Key areas are LLMs, computer vision, and robotics."
    return {"research_notes": notes, "next": "writer"}

# Agent 2: Writer
def writer_agent(state: GraphState) -> GraphState:
    print("---> In Writer Agent")
    notes = state.get("research_notes")
    draft = f"Draft based on research:\n{notes}\n\nConclusion: The field of AI is dynamic and full of opportunities."
    return {"draft": draft, "next": "reviewer"}

# Agent 3: Reviewer
def reviewer_agent(state: GraphState) -> GraphState:
    print("---> In Reviewer Agent")
    draft = state.get("draft")
    review = f"Review of the draft: '{draft[:30]}...'. The conclusion is a bit weak. Suggest adding specific examples."
    # In a real scenario, we might loop back to the writer, but for simplicity, we'll end here.
    return {"review": review, "next": "end"}

## 2. The Supervisor: Routing the Task

The core of a multi-agent system is the router or supervisor. This is a node that decides which agent should be called next based on the current state. We implement this with conditional edges in LangGraph.

In [ ]:
# The supervisor node doesn't do any work, it just decides the next step
def supervisor_node(state: GraphState) -> dict:
    # The first time this node is called, 'next' will be None
    if state.get("next") is None:
        return {"next": "researcher"}
    # Otherwise, the next step is determined by the last agent that ran
    return {"next": state.get("next")}

# Define the conditional mapping
conditional_map = {
    "researcher": "researcher_agent",
    "writer": "writer_agent",
    "reviewer": "reviewer_agent",
    "end": END
}

## 3. Assembling the Graph

Now, we wire everything together.

In [ ]:
workflow = StateGraph(GraphState)

# Add the worker agent nodes
workflow.add_node("researcher_agent", researcher_agent)
workflow.add_node("writer_agent", writer_agent)
workflow.add_node("reviewer_agent", reviewer_agent)

# Add the supervisor node
workflow.add_node("supervisor", supervisor_node)

# Set the entry point
workflow.set_entry_point("supervisor")

# Add the conditional edges from the supervisor to the workers
workflow.add_conditional_edges(
    "supervisor",
    lambda state: state["next"],
    conditional_map
)

# Add edges from the workers back to the supervisor
workflow.add_edge("researcher_agent", "supervisor")
workflow.add_edge("writer_agent", "supervisor")
workflow.add_edge("reviewer_agent", "supervisor")

# Compile the graph
multi_agent_app = workflow.compile()

print("Multi-agent graph compiled successfully!")

## 4. Running the Multi-Agent System

Let's give the team a task and watch the flow.

In [ ]:
initial_state = {"task": "Write a report on the future of AI"}

final_state = multi_agent_app.invoke(initial_state)

print("\n--- Final Result ---")
print(f"Task: {final_state['task']}")
print(f"Research: {final_state['research_notes']}")
print(f"Draft: {final_state['draft']}")
print(f"Review: {final_state['review']}")

## 🔧 Hands-On Exercise

**Goal:** Create a simple two-agent debate.

1.  Define a `DebateState` with fields for `topic`, `agent_one_argument`, `agent_two_argument`, and `next`.
2.  Create an `agent_one` node that provides an argument for the topic.
3.  Create an `agent_two` node that provides a counter-argument.
4.  Build a graph where the supervisor first calls agent one, then agent two, and then ends.
5.  Run the debate on the topic "Cats vs. Dogs".

In [ ]:
class DebateState(TypedDict):
    topic: str
    argument_one: str
    argument_two: str
    next: str

def agent_one(state: DebateState):
    print("---> Agent One (For)")
    return {"argument_one": f"Argument for {state['topic']}: They are independent and clean.", "next": "agent_two"}

def agent_two(state: DebateState):
    print("---> Agent Two (Against)")
    return {"argument_two": f"Counter-argument for {state['topic']}: They are loyal and playful.", "next": "end"}

def debate_supervisor(state: DebateState):
    return {"next": state.get("next", "agent_one")}

debate_graph = StateGraph(DebateState)
debate_graph.add_node("agent_one", agent_one)
debate_graph.add_node("agent_two", agent_two)
debate_graph.add_node("supervisor", debate_supervisor)
debate_graph.set_entry_point("supervisor")
debate_graph.add_conditional_edges("supervisor", lambda s: s['next'], {"agent_one": "agent_one", "agent_two": "agent_two", "end": END})
debate_graph.add_edge("agent_one", "supervisor")
debate_graph.add_edge("agent_two", "supervisor")

debate_app = debate_graph.compile()
final_debate_state = debate_app.invoke({"topic": "Cats"})
print(f"\n--- Debate Result ---\n{final_debate_state}")

## 💡 Pro Tip

For complex multi-agent systems, think of your graph's state as a shared whiteboard or a project management ticket. Each agent is a team member who reads the ticket, performs their task, and then updates the ticket with their results and passes it on to the next person. The supervisor is the project manager who assigns the ticket to the right team member at each stage.

## 🏁 Summary

You can now build entire teams of AI agents that work together!

1.  **Divide and Conquer:** Multi-agent systems excel at solving complex problems by breaking them down and assigning them to specialized agents.
2.  **The Supervisor is Key:** The power of this pattern comes from the supervisor node, which uses conditional logic to route tasks between the different agents.
3.  **The State is a Shared Workspace:** The `StateGraph`'s state object is the central place where agents share information and pass results to one another.